# 02. AGEVector — Vector Mode

pgvector 기반 벡터 검색. `Neo4jVector` / `PGVectorStore`와 동일한 인터페이스.

**사전 준비**: Docker 컨테이너 실행
```bash
cd docker && docker compose up -d
```

In [ ]:
DSN = "host=localhost port=5433 dbname=langchain_age user=langchain password=langchain"

## 임베딩 함수 설정

OpenAI, HuggingFace 등 어떤 임베딩이든 사용 가능. 여기서는 API 키 없이 동작하는 Fake 임베딩으로 시연.

In [ ]:
import hashlib

class DemoEmbeddings:
    """API 키 없이 동작하는 결정적 임베딩 (dim=64). 시연 전용."""
    def embed_documents(self, texts):
        return [self._hash_embed(t) for t in texts]

    def embed_query(self, text):
        return self._hash_embed(text)

    def _hash_embed(self, text, dim=64):
        h = hashlib.sha256(text.encode()).digest()
        # SHA-256 = 32 bytes → 64 float values
        return [b / 255.0 for b in h * (dim // len(h) + 1)][:dim]

embeddings = DemoEmbeddings()

# OpenAI를 쓰려면:
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings()

## 1. 기본 벡터 검색

In [ ]:
from langchain_age import AGEVector, DistanceStrategy

store = AGEVector(
    connection_string=DSN,
    embedding_function=embeddings,
    collection_name="demo_vectors",
    distance_strategy=DistanceStrategy.COSINE,
    pre_delete_collection=True,  # 깨끗한 시작
)

# 문서 추가
texts = [
    "Apache AGE adds graph query capabilities to PostgreSQL.",
    "pgvector enables fast vector similarity search in PostgreSQL.",
    "LangChain is a framework for building LLM applications.",
    "Neo4j is a native graph database using the Bolt protocol.",
    "PostgreSQL supports JSON, full-text search, and extensions.",
    "RAG combines retrieval and generation for accurate answers.",
    "GraphRAG uses knowledge graphs to enhance LLM context.",
    "Vector embeddings represent text as high-dimensional points.",
]

ids = store.add_texts(texts)
print(f"Added {len(ids)} documents.")

In [ ]:
# 유사도 검색
results = store.similarity_search("graph database PostgreSQL", k=3)
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 2. 점수와 함께 검색

In [ ]:
# 거리 점수 (낮을수록 유사)
results = store.similarity_search_with_score("vector search", k=3)
for doc, distance in results:
    print(f"  [{distance:.4f}] {doc.page_content}")

In [ ]:
# 관련도 점수 (0~1, 높을수록 유사)
results = store.similarity_search_with_relevance_scores("vector search", k=3)
for doc, score in results:
    print(f"  [{score:.4f}] {doc.page_content}")

## 3. 메타데이터 필터링

MongoDB 스타일 필터 연산자 지원: `$eq`, `$ne`, `$lt`, `$gt`, `$in`, `$between`, `$like`, `$and`, `$or` 등.

In [ ]:
# 메타데이터와 함께 문서 추가
store.add_texts(
    ["Python is great for data science.", "TypeScript is popular for web dev."],
    metadatas=[{"lang": "python", "year": 2024}, {"lang": "typescript", "year": 2024}],
)

# 필터 검색
results = store.similarity_search("programming", k=5, filter={"lang": "python"})
for doc in results:
    print(f"  {doc.page_content} | metadata: {doc.metadata}")

In [ ]:
# 복합 필터
results = store.similarity_search(
    "programming",
    k=5,
    filter={"$or": [{"lang": "python"}, {"lang": "typescript"}]},
)
for doc in results:
    print(f"  {doc.page_content}")

## 4. 하이브리드 검색 (Vector + Full-text)

벡터 유사도와 PostgreSQL 전문 검색을 RRF(Reciprocal Rank Fusion)로 결합.

In [ ]:
from langchain_age import SearchType

hybrid_store = AGEVector(
    connection_string=DSN,
    embedding_function=embeddings,
    collection_name="demo_vectors",  # 같은 테이블 재사용
    search_type=SearchType.HYBRID,
)

# 하이브리드 검색: 벡터 유사도 + 키워드 매칭
results = hybrid_store.similarity_search("PostgreSQL graph", k=3)
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 5. MMR (Maximal Marginal Relevance)

관련성과 다양성의 균형을 맞추는 검색. DB에 저장된 임베딩을 재사용하므로 추가 API 호출 없음.

In [ ]:
results = store.max_marginal_relevance_search(
    "database technology",
    k=3,
    fetch_k=8,
    lambda_mult=0.5,  # 0=최대 다양성, 1=최대 유사성
)
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 6. HNSW 인덱스 생성

대규모 데이터에서 근사 최근접 이웃 검색 성능 향상.

In [ ]:
store.create_hnsw_index(m=16, ef_construction=64)
print("HNSW index created.")

# 인덱스 후에도 동일하게 검색
results = store.similarity_search("graph database", k=2)
for doc in results:
    print(f"  {doc.page_content}")

## 7. LangChain Retriever로 사용

In [ ]:
retriever = store.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is RAG?")
for doc in docs:
    print(f"  {doc.page_content}")

## 8. 정리

In [ ]:
store.drop_index()
store._drop_table()
store.close()
print("Done.")